In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/urdu-ocr-si26'
os.chdir(PROJECT_PATH)
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/urdu-ocr-si26


## Week 3 — Dataset Expansion & PyTorch Dataset Class

This notebook expands the Urdu OCR dataset from 101 to 201 images (adding synthetic
and existing-dataset images for variety), and builds a PyTorch Dataset class that
wraps the labeled images using the TrOCR processor — preparing the data pipeline
for model training in Week 4.

In [2]:
!pip install transformers torch pillow pandas --quiet

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path, encoding='utf-8-sig')
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        # Load and convert image
        image = Image.open(row['image']).convert('RGB')
        # Process image for the model
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        # Process the text label
        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

print('Dataset class defined!')

Dataset class defined!


In [3]:
# Load the TrOCR processor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# Create dataset
dataset = UrduOCRDataset('data/labels.csv', processor)

# Test it loads correctly
sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

# Create train / test split (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Dataset loaded: 201 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 41


In [ ]:
!pip install sentencepiece protobuf --quiet
import os
os.kill(os.getpid(), 9)

In [3]:
try:
    import sentencepiece
    print("sentencepiece VERSION:", sentencepiece.__version__)
except ImportError as e:
    print("NOT INSTALLED:", e)

sentencepiece VERSION: 0.2.2


In [4]:
from transformers import TrOCRProcessor
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
print("Processor loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
!pip install --upgrade transformers --quiet
!rm -rf /root/.cache/huggingface

import os
os.kill(os.getpid(), 9)

In [1]:
!pip install -q transformers==4.41.0 tokenizers==0.19.1 sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
!pip install -q transformers==4.41.0 tokenizers==0.19.1 sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
from getpass import getpass
import subprocess, shutil, os

TOKEN = getpass("GitHub token paste karein aur Enter dabayein: ")
USERNAME = "waroodzahrakhan"
REPO = "urdu-ocr-codesaviours-si26-warood"

repo_url = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

os.chdir('/content')
if os.path.exists('repo_upload'):
    shutil.rmtree('repo_upload')
os.system(f'git clone {repo_url} repo_upload')

os.chdir('/content/repo_upload')

# Notebook ko Drive se dhoondein aur copy karein
notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week3-Warood.ipynb'
notebook_dest = '/content/repo_upload/SI26-Week3-Warood.ipynb'
shutil.copy(notebook_source, notebook_dest)
print("Notebook copy ho gaya")

subprocess.run(['git', 'config', 'user.email', 'warood@example.com'])
subprocess.run(['git', 'config', 'user.name', 'waroodzahrakhan'])
subprocess.run(['git', 'add', 'SI26-Week3-Warood.ipynb'])

commit_result = subprocess.run(['git', 'commit', '-m', 'Add Week 3 notebook'], capture_output=True, text=True)
print("COMMIT:", commit_result.stdout, commit_result.stderr)

push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)
print("PUSH RETURN CODE:", push_result.returncode)

GitHub token paste karein aur Enter dabayein: ··········
Notebook copy ho gaya
COMMIT: [main 2289259] Add Week 3 notebook
 1 file changed, 1 insertion(+)
 create mode 100644 SI26-Week3-Warood.ipynb
 
PUSH STDOUT: 
PUSH STDERR: To https://github.com/waroodzahrakhan/urdu-ocr-codesaviours-si26-warood.git
   cc9d81b..2289259  main -> main

PUSH RETURN CODE: 0


In [4]:
!pip install -q transformers==4.41.0 tokenizers==0.19.1 sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import os
os.kill(os.getpid(), 9)

In [4]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > GPU')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

print('Model loaded successfully!')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Using device: cuda


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model parameters: 333,921,792


In [5]:
from torch.utils.data import DataLoader
from transformers import AdamW

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)

# Optimiser
optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches per epoch: 40
Ready to train!


In [6]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/3
------------------------------
  Batch 0/40 | Loss: 17.8888


FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/synthetic/synthetic_077.png'

In [7]:
import os
print("Current directory:", os.getcwd())
print("File exists?", os.path.exists('data/raw/synthetic/synthetic_077.png'))
print("Synthetic folder contents:", os.listdir('data/raw/synthetic')[:10])

Current directory: /content/drive/MyDrive/urdu-ocr-si26
File exists? False
Synthetic folder contents: ['synthetic_020.png', 'synthetic_017.png', 'synthetic_018.png', 'synthetic_019.png', 'synthetic_016.png', 'synthetic_003.png', 'synthetic_001.png', 'synthetic_002.png', 'synthetic_004.png', 'synthetic_005.png']


In [6]:
import os
print(os.path.exists('data/raw/synthetic/synthetic_077.png'))
print(os.listdir('data/raw/synthetic')[:15])

False
['synthetic_020.png', 'synthetic_017.png', 'synthetic_018.png', 'synthetic_019.png', 'synthetic_016.png', 'synthetic_003.png', 'synthetic_001.png', 'synthetic_002.png', 'synthetic_004.png', 'synthetic_005.png', 'synthetic_006.png', 'synthetic_007.png', 'synthetic_009.png', 'synthetic_008.png', 'synthetic_010.png']


In [7]:
import pandas as pd
import os

df = pd.read_csv('data/labels.csv', encoding='utf-8-sig')
missing = [img for img in df['image'] if not os.path.exists(img)]
print(f"Total labels: {len(df)}")
print(f"Missing files: {len(missing)}")
print("First 10 missing:", missing[:10])

# Kitni actual synthetic images hain
synthetic_files = os.listdir('data/raw/synthetic')
print(f"\nSynthetic images actually present: {len(synthetic_files)}")
print(sorted(synthetic_files))

Total labels: 201
Missing files: 10
First 10 missing: ['data/raw/synthetic/synthetic_071.png', 'data/raw/synthetic/synthetic_072.png', 'data/raw/synthetic/synthetic_073.png', 'data/raw/synthetic/synthetic_074.png', 'data/raw/synthetic/synthetic_075.png', 'data/raw/synthetic/synthetic_076.png', 'data/raw/synthetic/synthetic_077.png', 'data/raw/synthetic/synthetic_078.png', 'data/raw/synthetic/synthetic_079.png', 'data/raw/synthetic/synthetic_080.png']

Synthetic images actually present: 70
['synthetic_001.png', 'synthetic_002.png', 'synthetic_003.png', 'synthetic_004.png', 'synthetic_005.png', 'synthetic_006.png', 'synthetic_007.png', 'synthetic_008.png', 'synthetic_009.png', 'synthetic_010.png', 'synthetic_011.png', 'synthetic_012.png', 'synthetic_013.png', 'synthetic_014.png', 'synthetic_015.png', 'synthetic_016.png', 'synthetic_017.png', 'synthetic_018.png', 'synthetic_019.png', 'synthetic_020.png', 'synthetic_021.png', 'synthetic_022.png', 'synthetic_023.png', 'synthetic_024.png', '

In [8]:
import pandas as pd
import os

df = pd.read_csv('data/labels.csv', encoding='utf-8-sig')

# Sirf woh rows rakho jinki image file exist karti hai
df_clean = df[df['image'].apply(os.path.exists)].reset_index(drop=True)
print(f"Original: {len(df)} | After removing missing: {len(df_clean)}")

# Clean version save karo
df_clean.to_csv('data/labels_clean.csv', index=False, encoding='utf-8-sig')
print("Saved as data/labels_clean.csv")

Original: 201 | After removing missing: 191
Saved as data/labels_clean.csv


In [9]:
dataset = UrduOCRDataset('data/labels_clean.csv', processor)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)

print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 191 samples
Training samples: 152
Testing samples: 39


In [10]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4)
print(f'Training batches per epoch: {len(train_loader)}')

Training batches per epoch: 38


In [11]:
from transformers import AdamW
optimizer = AdamW(model.parameters(), lr=5e-5)
print('Optimizer ready!')

Optimizer ready!


/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:588: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [12]:
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/3
------------------------------
  Batch 0/38 | Loss: 17.7695
  Batch 10/38 | Loss: 1.4213
  Batch 20/38 | Loss: 0.6585
  Batch 30/38 | Loss: 2.0630
Epoch 1 complete | Average Loss: 2.0618

Epoch 2/3
------------------------------
  Batch 0/38 | Loss: 1.9020
  Batch 10/38 | Loss: 0.7434
  Batch 20/38 | Loss: 0.8003
  Batch 30/38 | Loss: 1.0011
Epoch 2 complete | Average Loss: 0.9858

Epoch 3/3
------------------------------
  Batch 0/38 | Loss: 0.8896
  Batch 10/38 | Loss: 0.7456
  Batch 20/38 | Loss: 0.5477
  Batch 30/38 | Loss: 0.9348
Epoch 3 complete | Average Loss: 0.8349

Training complete!


In [13]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )
        actual_text = processor.batch_decode(
            labels, skip_special_tokens=True
        )

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            print(f'Predicted: {pred}')
            print(f'Actual: {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

=== Model Evaluation on Test Images ===



/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Predicted: ا�ااا��
Actual: اباما

Predicted: ا�ا،���
Actual: اجنتا

Predicted: ����������ڌ������
Actual: ناقابلِ قبول

Predicted: ا��ٌ���
Actual: آیئے

Predicted: ا�ا�ٌ��
Actual: آڈھا

Predicted: ا�ا�ا،�
Actual: اجنکا

Predicted: ا��ٌ���
Actual: اتم

Predicted: ا�ا�ٌ��
Actual: اجود

Predicted: ا�ا�ا،�
Actual: اخترخان

Predicted: ا�ا�ا���
Actual: اتھاہ

Predicted: ������������������
Actual: اب مجھے زندگی سے کوئی شکایت نہیں

Predicted: ا�ا�ٌ���
Actual: ادو

Predicted: ������������������
Actual: پھر اسے بھول جاؤ

Predicted: ا�ا�ا���
Actual: ءالدین

Predicted: اااااا����
Actual: تجارتی جہاز

Predicted: ا�ٌ�����
Actual: آئیڈیا

Predicted: ������������������
Actual: ہر مشکل کا حل موجود ہوتا ہے

Predicted: ا�ا��ٌ�
Actual: ابراھیم

Predicted: ������������������
Actual: وہ خاموشی سے دیکھتی رہی

Predicted: ا��ٌ���
Actual: آبلے

Predicted: ������������������
Actual: پہاڑ بہت اونچے تھے

Predicted: ������������������
Actual: غریبوں کی مدد کرنی چاہیے

Predicted: ا���ٌ�
Actual: آبادپر

Predict

In [14]:
from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/SI26-urdu-ocr-model'
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print(f'Model saved to Google Drive: {save_path}')
print('You can load this model again next week without retraining')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to Google Drive: /content/drive/MyDrive/SI26-urdu-ocr-model
You can load this model again next week without retraining


In [15]:
from getpass import getpass
import subprocess, shutil, os

TOKEN = getpass("GitHub token paste karein aur Enter dabayein: ")
USERNAME = "waroodzahrakhan"
REPO = "urdu-ocr-codesaviours-si26-warood"

repo_url = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

os.chdir('/content')
if os.path.exists('repo_upload2'):
    shutil.rmtree('repo_upload2')
os.system(f'git clone {repo_url} repo_upload2')

os.chdir('/content/repo_upload2')

notebook_source = '/content/drive/MyDrive/Colab Notebooks/SI26-Week3-Warood.ipynb'
notebook_dest = '/content/repo_upload2/SI26-Week4-Warood.ipynb'
shutil.copy(notebook_source, notebook_dest)
print("Notebook copy ho gaya")

subprocess.run(['git', 'config', 'user.email', 'warood@example.com'])
subprocess.run(['git', 'config', 'user.name', 'waroodzahrakhan'])
subprocess.run(['git', 'add', 'SI26-Week4-Warood.ipynb'])

commit_result = subprocess.run(['git', 'commit', '-m', 'Add Week 4 notebook (training+eval, includes Week 3 dataset code)'], capture_output=True, text=True)
print("COMMIT:", commit_result.stdout, commit_result.stderr)

push_result = subprocess.run(['git', 'push'], capture_output=True, text=True)
print("PUSH STDOUT:", push_result.stdout)
print("PUSH STDERR:", push_result.stderr)

KeyboardInterrupt: Interrupted by user